# Case Study: Brazilian Central Bank Regulations

**Dataset:** `BrazilianRegulatoryDataset` (20 hand-crafted QA examples)
**Sources:** Basel III (BIS) and BCB Resolution 4.658 (cybersecurity)


## Why regulated-market QA is harder

Open-domain QA benchmarks reward breadth; regulated-market QA rewards
*refusal calibration*. A model that confidently hallucinates the CET1
ratio is strictly worse than one that says 'I don't know' — the second
can be routed to a human, the first gets a bank sanctioned.

Three structural problems make this domain hard:

1. **Training-data scarcity.** Basel III technical annexes and BCB
   Portuguese-language circulars are under-represented in web-scale
   pretraining corpora.
2. **High stakes.** A wrong answer on a capital-adequacy question can
   drive real capital allocation decisions.
3. **Refusal must be calibrated.** A blanket 'I don't know' is useless;
   the model must refuse *exactly where it is wrong*.

This notebook uses `DummyBackend` so it runs anywhere (including CI)
with no GPU. The structure is identical for real backends — swap
`backend='dummy'` for `'hf'` or `'openai'` and re-run.

In [ ]:
from lub.benchmarks import BenchmarkRunner, BrazilianRegulatoryDataset
from lub.calibration.plots import plot_reliability_diagram
from lub.pipeline import UncertaintyPipeline
from lub.reports.renderer import AIRMFReporter

## 1. Build a pipeline

`DummyBackend` is deterministic: its 'answers' are hash-derived
strings. We use `token_logprob` as the estimator because it's the
cheapest single-pass baseline; for a real model we would swap in
`semantic_entropy` or `self_consistency`.

In [ ]:
pipe = UncertaintyPipeline.from_pretrained(
    model='dummy-regulatory',
    backend='dummy',
    estimator='token_logprob',
    refusal_threshold=0.5,
)
dataset = BrazilianRegulatoryDataset()
print(f'Dataset: {dataset.name} v{dataset.version}')
print(f'Dataset hash: {dataset.hash()[:16]}...')

## 2. Run the benchmark

We disable JSON writing (`write=False`) because this notebook is
demo code and we don't want to pollute `benchmarks/results/`.

In [ ]:
runner = BenchmarkRunner(pipeline=pipe, dataset=dataset)
result = runner.run(limit=20, seed=0, write=False)
print(f'n={result.n}  accuracy={result.accuracy:.3f}  ece={result.ece:.3f}  auroc={result.refusal_auroc:.3f}')

## 3. Results table

Per-example view of what the pipeline predicted, its confidence, and
whether the refusal gate fired.

In [ ]:
rows = []
for ex in dataset.load():
    r = pipe.answer(ex.question)
    rows.append({
        'id': ex.id,
        'topic': ex.metadata.get('topic', ''),
        'gold': ex.gold_answer,
        'pred': r.answer[:32],
        'confidence': round(r.confidence, 3),
        'refused': r.should_refuse,
    })
try:
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df)
except ImportError:
    for row in rows[:5]:
        print(row)

## 4. Reliability diagram

The diagonal is perfect calibration. Points above → under-confident;
points below → over-confident. With the dummy backend everything
collapses to a single bin because all confidences are identical.

In [ ]:
confs = [row['confidence'] for row in rows]
correct = [1.0 if row['pred'].strip().lower() == row['gold'].strip().lower() else 0.0 for row in rows]
fig = plot_reliability_diagram(confs, correct, n_bins=10, title='BR regulatory — dummy backend')
fig

## 5. AI RMF report

Render an auditor-facing report for this single run. In production
you would collect multiple `BenchmarkResult` records (one per
dataset × estimator × model) and render them together.

In [ ]:
reporter = AIRMFReporter(results=[result], title='BR Regulatory — Dummy Baseline')
markdown = reporter.render(format='md')
print(markdown[:1200])

## 6. Discussion — what an examiner would see

An OCC or Fed examiner reviewing this report asks three questions:

1. **Is the calibration plausible?** ECE near 0 on 20 examples is not
   evidence of a well-calibrated system — the sample is too small.
   In real deployment we would need at least 500–1000 examples per
   regulatory topic to have tight confidence intervals on ECE.
2. **Does the refusal gate separate right from wrong?** The
   `refusal_auroc` column answers this. For dummy it's 0.5 (random),
   which is correct — the dummy is not a real model.
3. **Is the evidence reproducible?** Every result record carries the
   dataset hash, git SHA, and `package_versions` dict, so the
   examiner can rerun with `lub repro` and verify the numbers.

**What would change for production.** Replace `DummyBackend` with an
on-prem `HFBackend` (bank policy rarely allows SaaS inference for
regulatory QA), switch the estimator to `semantic_entropy` for
better calibration on open-ended answers, and re-run over the full
BCB + Basel corpus — not just the 20 hand-crafted smoke questions.